In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
db_path = Path(r"E:\ProyectoAnalisisElectrico\DiaPromedio\Mensuales")
save_path = Path(r"E:\ProyectoAnalisisElectrico\DiaPromedio\Periodos")


In [3]:
month_mapping = {
    "enero": "01", "febrero": "02", "marzo": "03", "abril": "04",
    "mayo": "05", "junio": "06", "julio": "07", "agosto": "08",
    "septiembre": "09", "octubre": "10", "noviembre": "11", "diciembre": "12"
}

def get_folders_by_period(start_date=None, end_date=None, meses_filtro=None):
    available_folders = sorted([d.name for d in db_path.iterdir()])
    
    if start_date and end_date:
        start_str = f"{start_date[1] % 100:02d}{month_mapping[start_date[0]]}"
        end_str = f"{end_date[1] % 100:02d}{month_mapping[end_date[0]]}"
        idx_start = available_folders.index(start_str)
        idx_end = available_folders.index(end_str)
        available_folders = available_folders[idx_start : idx_end + 1]
        
    if meses_filtro:
        codigos_permitidos = [month_mapping[m.lower()] for m in meses_filtro]
        available_folders = [f for f in available_folders if f[-2:] in codigos_permitidos]
        
    return available_folders

def build_dynamic_mask(date_column, periods_list):
    dates_str = date_column.astype(str)
    formatted_dates = dates_str.str[2:4] + dates_str.str[5:7]
    client_dates = set(formatted_dates)
    return "".join("1" if period in client_dates else "0" for period in periods_list)

In [ ]:
estaciones_meses = {
    "Verano": ["enero", "febrero", "marzo"],
    "Otono": ["abril", "mayo", "junio"],
    "Invierno": ["julio", "agosto", "septiembre"],
    "Primavera": ["octubre", "noviembre", "diciembre"]
}
periods = get_folders_by_period(
    start_date=("mayo", 2025), 
    end_date=("abril", 2026), 
    meses_filtro=estaciones_meses["Invierno"]
)

print(f"Carpetas seleccionadas: {periods}")

dfs = []
for month in periods:
    df_path = db_path / month / f"{month}_mean_month.parquet"
    dfs.append(pd.read_parquet(df_path))
    
combined_df = pd.concat(dfs, ignore_index=True)

period_label = f"{periods[0]}_{periods[-1]}_invierno"
combined_df["period"] = period_label

Carpetas seleccionadas: ['2507', '2508', '2509']


In [21]:
# 1. Reconstrucción matemática de sumas y cuadrados antes del GroupBy
metrics = ['medida', 'CMg[CLP/KWh]', 'valorizado_CLP']
for m in metrics:
    combined_df[f'{m}_sum_val'] = combined_df[f'{m}_mean'] * combined_df[f'{m}_count']
    
    # fillna(0) previene errores cuando count es 1 y std es NaN
    std_squared = combined_df[f'{m}_std'].fillna(0) ** 2
    combined_df[f'{m}_sum_sq'] = (combined_df[f'{m}_count'] - 1) * std_squared + combined_df[f'{m}_count'] * (combined_df[f'{m}_mean'] ** 2)

sep_char = "::"
agg_rules = {
    # --- METRICS ---
    'medida_sum_val': ('medida_sum_val', 'sum'),
    'medida_sum_sq': ('medida_sum_sq', 'sum'),
    'medida_count': ('medida_count', 'sum'),
    
    'CMg_sum_val': ('CMg[CLP/KWh]_sum_val', 'sum'),
    'CMg_sum_sq': ('CMg[CLP/KWh]_sum_sq', 'sum'),
    'CMg_count': ('CMg[CLP/KWh]_count', 'sum'),
    
    'valorizado_sum_val': ('valorizado_CLP_sum_val', 'sum'),
    'valorizado_sum_sq': ('valorizado_CLP_sum_sq', 'sum'),
    'valorizado_count': ('valorizado_CLP_count', 'sum'),
    
    # --- CALENDAR ---
    'active_calendar': ('Año_Mes', lambda x: build_dynamic_mask(x, periods)),
    
    # --- RUT ---
    'RUT': ('RUT', 'last'),
    'rut_log': ('RUT', lambda x: sep_char.join(x.dropna().astype(str).unique())),
    'n_ruts': ('RUT', 'nunique'), 

    # --- RAZON SOCIAL ---
    'Razon_Social': ('Razon_Social', 'last'),
    'razon_social_log': ('Razon_Social', lambda x: sep_char.join(x.dropna().astype(str).unique())),
    'n_razones_sociales': ('Razon_Social', 'nunique'), 
    
    # --- NOMBRE CORTO ---
    'Nombre_Corto': ('Nombre_Corto', 'last'),
    'nombre_corto_log': ('Nombre_Corto', lambda x: sep_char.join(x.dropna().astype(str).unique())),
    'n_nombres_cortos': ('Nombre_Corto', 'nunique'), 
    
    # --- BARRA ---
    'nombre_barra': ('nombre_barra', 'last'),
    'nombre_barra_log': ('nombre_barra', lambda x: sep_char.join(x.dropna().astype(str).unique())),
    'n_nombres_barra': ('nombre_barra', 'nunique'), 
    
    # --- TENSION ---
    'tension': ('tension', 'last'),
    'tension_log': ('tension', lambda x: sep_char.join(x.dropna().astype(str).unique())),
    'n_tensiones': ('tension', 'nunique'), 
    
    # --- OTROS ---
    'tipo':  ('tipo', 'last'),
    'period': ('period', 'last')
}

group_cols = ['clave', 'Zona', 'Hora']

final_df = combined_df.groupby(group_cols).agg(**agg_rules).reset_index()

In [22]:
# 2. Cálculo final de Promedios y Desviaciones Estándar Combinadas
prefix_map = ['medida', 'CMg', 'valorizado']
for m, prefix in zip(metrics, prefix_map):
    # Promedio Combinado
    final_df[f'{m}_mean'] = final_df[f'{prefix}_sum_val'] / final_df[f'{prefix}_count']
    
    # Varianza Combinada y Desviación Estándar
    variance = (final_df[f'{prefix}_sum_sq'] - (final_df[f'{prefix}_sum_val'] ** 2 / final_df[f'{prefix}_count'])) / (final_df[f'{prefix}_count'] - 1)
    
    # np.maximum evita raíces cuadradas negativas originadas por precisión de punto flotante
    final_df[f'{m}_std'] = np.sqrt(np.maximum(0, variance))
    
    # Restauramos el conteo con el nombre correcto
    final_df[f'{m}_count'] = final_df[f'{prefix}_count']
    
    # Limpieza de columnas temporales de cálculo
    final_df = final_df.drop(columns=[f'{prefix}_sum_val', f'{prefix}_sum_sq', f'{prefix}_count'])

# Total de energía usando el nuevo nombre de la columna promedio
final_df['medida_total'] = final_df.groupby(['clave', 'Zona'])['medida_mean'].transform('sum')



In [23]:
save_folder = save_path / period_label
if save_folder.is_dir():
    raise RuntimeError(f"Data for {period_label} already processed. Skipping...")
save_folder.mkdir(parents=True, exist_ok=True)

final_df.to_parquet(save_folder / f"{period_label}_mean_period.parquet", engine="pyarrow", compression="snappy")